<a href="https://colab.research.google.com/github/OlaStrzelczyk/Non-relational_database_project/blob/main/Projekt_AOKO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Automatyczna klasyfikacja zmian nowotworowych na obrazach ultrasonograficznych do klas: normal, benign, malignant**

In [96]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [97]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [98]:
dataset_path = "/content/drive/MyDrive/breastCancer"

In [99]:
os.listdir(dataset_path)

['malignant', 'normal', 'benign']

In [100]:
for category in os.listdir(dataset_path):
    path = os.path.join(dataset_path, category)
    print(category, len(os.listdir(path)))

malignant 210
normal 133
benign 437


In [101]:
img_size = (224, 224)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

Found 780 files belonging to 3 classes.
Using 624 files for training.
Found 780 files belonging to 3 classes.
Using 156 files for validation.


In [102]:
class_names = train_ds.class_names
print(class_names)

['benign', 'malignant', 'normal']


AUGUMENTACJA

In [103]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

EfficientNet

In [104]:
base_model = tf.keras.applications.EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

MODEL

In [105]:
inputs = keras.Input(shape=(224,224,3))

x = data_augmentation(inputs)

x = tf.keras.applications.efficientnet.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(3, activation='softmax')(x)

model = keras.Model(inputs, outputs)

KOMPILACJA

In [106]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=[
        'accuracy'
    ]
)

CALLBACKI

In [107]:
callbacks = [

    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2
    ),

    keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor='val_accuracy',
        save_best_only=True
    )

]

CALSS_WEIGHTS

In [108]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = np.concatenate([
    y.numpy() for x, y in train_ds
])

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

{0: np.float64(0.5909090909090909), 1: np.float64(1.2530120481927711), 2: np.float64(1.9622641509433962)}


TRENING

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    class_weight=class_weights
)

Epoch 1/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 95s 4s/step - accuracy: 0.4647 - loss: 1.0418 - val_accuracy: 0.5962 - val_loss: 0.8811 - learning_rate: 0.0010
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 75s 4s/step - accuracy: 0.5481 - loss: 0.9165 - val_accuracy: 0.6410 - val_loss: 0.8020 - learning_rate: 0.0010
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 83s 4s/step - accuracy: 0.6042 - loss: 0.8312 - val_accuracy: 0.6154 - val_loss: 0.8082 - learning_rate: 0.0010
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 83s 4s/step - accuracy: 0.6218 - loss: 0.7881 - val_accuracy: 0.6474 - val_loss: 0.7506 - learning_rate: 0.0010
Epoch 5/20
 5/20 ━━━━━━━━━━━━━━━━━━━━ 44s 3s/step - accuracy: 0.6837 - loss: 0.6025

matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

# prawdziwe etykiety
y_true = np.concatenate([
    y for x, y in val_ds
], axis=0)

# predykcje modelu
y_pred = np.argmax(
    model.predict(val_ds),
    axis=1
)

# confusion matrix
cm = confusion_matrix(y_true, y_pred)

# wyświetlenie
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(8,8))

disp.plot(ax=ax)

plt.title("Confusion Matrix")

plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

WYKRESY

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.legend(['train', 'val'])

plt.title("Accuracy")

plt.show()

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

rekompilacja

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)